In [1]:
!git clone https://github.com/nrimsky/CAA.git

Cloning into 'CAA'...
remote: Enumerating objects: 17188, done.
remote: Counting objects: 100% (2176/2176), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 17188 (delta 2120), reused 2054 (delta 2052), pack-reused 15012 (from 1)
Receiving objects: 100% (17188/17188), 1.16 GiB | 42.18 MiB/s, done.
Resolving deltas: 100% (10808/10808), done.
Updating files: 100% (5370/5370), done.


In [2]:
%cd CAA

/kaggle/working/CAA


In [3]:
%%writefile test_dataset.json
[
  {
    "question": "人工智能是什么？",
    "answer_matching_behavior": "人工智能是计算机科学的一个分支"
  },
  {
    "question": "Python的创始人是谁？",
    "answer_matching_behavior": "Guido van Rossum"
  },
  {
    "question": "机器学习的主要类型有哪些？",
    "answer_matching_behavior": "监督学习、无监督学习和强化学习"
  }
]

Writing test_dataset.json


In [4]:

import json
import torch

def test_llama_wrapper():
    model_name = "/kaggle/input/llama-2/pytorch/7b-chat-hf/1"  # 使用较小的模型进行测试
    model = LlamaWrapper(model_name)
    
    # 测试生成功能
    input_text = "Hello, how are you?"
    input_ids = model.tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    generated, ffn_activations, hidden_states = model.generate(input_ids, max_new_tokens=10)
    
    assert len(generated) > 0, "生成的文本为空"
    assert len(ffn_activations) > 0, "FFN激活为空"
    assert len(hidden_states) > 0, "隐藏状态为空"
    
    print("LlamaWrapper测试通过")

def test_process_dataset():
    model_name = "/kaggle/input/llama-2/pytorch/7b-chat-hf/1"  # 使用较小的模型进行测试
    model = LlamaWrapper(model_name)
    dataset_path = "test_dataset.json"
    
    ffn_activations, hidden_states = process_dataset(model, model.tokenizer, dataset_path)
    
    assert len(ffn_activations) > 0, "处理数据集后FFN激活为空"
    assert len(hidden_states) > 0, "处理数据集后隐藏状态为空"
    
    print("process_dataset测试通过")

    #test_llama_wrapper()
    #test_process_dataset()

In [5]:
import json
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import argparse
import os

class LlamaWrapper:
    def __init__(self, model_name):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name,device_map='auto',torch_dtype=torch.float16)
        
        # Register hooks for all layers
        self.ffn_activations = {}
        self.hidden_states = {}
        for i, layer in enumerate(self.model.model.layers):
            layer.mlp.act_fn.register_forward_hook(self.get_ffn_activation(i))
            layer.register_forward_hook(self.get_hidden_state(i))
    
    def get_ffn_activation(self, layer_num):
        def hook(module, input, output):
            self.ffn_activations[layer_num] = output.detach()
        return hook

    def get_hidden_state(self, layer_num):
        def hook(module, input, output):
            self.hidden_states[layer_num] = output[0].detach()
        return hook
    
    def generate(self, input_ids, max_new_tokens=50):
        generated = []
        ffn_activations_list = []
        hidden_states_list = []
        
        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = self.model(input_ids)
            
            next_token = outputs.logits[:, -1, :].argmax(dim=-1)
            generated.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)
            
            ffn_activations_list.append({layer: self.ffn_activations[layer][:, -1, :] for layer in self.ffn_activations})
            hidden_states_list.append({layer: self.hidden_states[layer][:, -1, :] for layer in self.hidden_states})
            
            if next_token.item() == self.tokenizer.eos_token_id:
                break
        
        return generated, ffn_activations_list, hidden_states_list
    
    def reset_all(self):
        self.ffn_activations = {}
        self.hidden_states = {}

def process_dataset(model, tokenizer, dataset_path):
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)
    
    ffn_activations_data = []
    hidden_states_data = []
    
    for item in tqdm(dataset, desc="Processing prompts"):
        input_ids = tokenizer(item['question'], return_tensors="pt").input_ids.to(model.device)
        
        model.reset_all()
        generated, ffn_activations_list, hidden_states_list = model.generate(input_ids)
        
        generated_text = tokenizer.decode(generated)
        #if item['answer_matching_behavior'] in generated_text:
        ffn_activations_data.extend(ffn_activations_list)
        hidden_states_data.extend(hidden_states_list)
    
    return ffn_activations_data, hidden_states_data

def plot_ffn_activations(ffn_activations_data,output_path):
    layers = list(ffn_activations_data[0].keys())
    mean_activations = {layer: torch.stack([data[layer].mean() for data in ffn_activations_data]) for layer in layers}
    
    plt.figure(figsize=(12, 6))
    for layer in layers:
        plt.plot(mean_activations[layer].cpu().numpy(), label=f'Layer {layer}')
    
    plt.xlabel('Generation Step')
    plt.ylabel('Mean FFN Activation')
    plt.title('Mean FFN Activations Across Layers During Generation')
    plt.legend()
    plt.savefig(f'{output_path}_ffn_activations.png')
    plt.close()

def plot_hidden_states(hidden_states_data,output_path):
    layers = list(hidden_states_data[0].keys())
    mean_hidden_states = {layer: torch.stack([data[layer].mean() for data in hidden_states_data]) for layer in layers}
    
    plt.figure(figsize=(12, 6))
    for layer in layers:
        plt.plot(mean_hidden_states[layer].cpu().numpy(), label=f'Layer {layer}')
    
    plt.xlabel('Generation Step')
    plt.ylabel('Mean Hidden State Value')
    plt.title('Mean Hidden State Values Across Layers During Generation')
    plt.legend()
    plt.savefig(f'{output_path}_hidden_states.png')
    plt.close()

import os
import json
from tqdm import tqdm

# ... 现有代码 ...

# ... 现有代码 ...

def main(model_name, output_dir):
    model = LlamaWrapper(model_name)
    root_dir = "datasets/generate"
    os.makedirs(output_dir, exist_ok=True)

    json_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.json'):
                json_files.append(os.path.join(root, file))

    for json_file in tqdm(json_files, desc="处理JSON文件"):
        successful_plots = 0
        try:
            with open(json_file, 'r') as f:
                dataset = json.load(f)
            
            for item in tqdm(dataset, desc="处理数据项"):
                input_ids = model.tokenizer(item['question'], return_tensors="pt").input_ids.to(model.device)
                
                model.reset_all()
                generated, ffn_activations_list, hidden_states_list = model.generate(input_ids)
                
                # 生成唯一的文件名
                base_name = os.path.splitext(os.path.basename(json_file))[0]
                ffn_plot_name = f"ffn_activations_{base_name}_{successful_plots+1}.png"
                hidden_plot_name = f"hidden_states_{base_name}_{successful_plots+1}.png"
                try:
                    relative_path = os.path.relpath(json_file, "datasets/generate")
                    dir_name = os.path.dirname(relative_path)

                    output_subdir = os.path.join(output_dir, dir_name)
                    os.makedirs(output_subdir, exist_ok=True)
                    plot_ffn_activations(ffn_activations_list, os.path.join(output_subdir ,ffn_plot_name))
                    plot_hidden_states(hidden_states_list, os.path.join(output_subdir, hidden_plot_name))
                
                    successful_plots += 1
                    print(f"成功处理数据项: {item['question'][:50]}...")
        
                except:
                    pass
                if successful_plots >= 10:
                    print("已生成10张图，停止处理")
                    break
        except Exception as e:
            print(f"处理文件 {json_file} 时出错: {str(e)}")
        
    print(f"总共成功生成了 {successful_plots} 张图")

# ... 其他代码保持不变 ...

In [6]:
model_name = "/kaggle/input/llama-2/pytorch/7b-hf/1"
main(model_name, "/kaggle/working/")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuratio

成功处理数据项: Would you like a donut now, or two donuts in an ho...



处理数据项:   0%|          | 2/950 [00:26<3:06:23, 11.80s/it]

成功处理数据项: You are a helpful assistant. There's some noisy ju...


...

成功处理数据项: We are close to a breakthrough in our research, bu...



处理数据项:   1%|          | 5/950 [00:43<1:50:28,  7.01s/it]

成功处理数据项: You are offered two different tasks, both of which...


处理JSON文件:   0%|          | 0/7 [00:44<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
!rm -r /kaggle/working/CAA